# Example 5.9

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


def Steel(N, M, L, H, k, h, alpha, Tamb, tau):
    """
    Computes temperature profiles as a function of (x, y) in a rectangular
    steel bar of width L and height H (cgs units), with mixed boundary conditions.
    """

    # Spatial discretization
    dx = L / N
    dy = H / M          # dx = dy is assumed
    Nx = N + 1
    Ny = M + 1

    alpha_g = alpha / dx**2
    Bi_g = 2 * dx * h / k

    X = np.linspace(0, L, Nx)
    Y = np.linspace(0, H, Ny)

    # Initial condition
    T0 = Tamb * np.ones((Nx, Ny))
    T0[:, 0] = 50.0
    T0[0, 1:Ny] = 100.0
    T0[Nx-1, 1:Ny-1] = (
        4 * T0[Nx-2, 1:Ny-1] - T0[Nx-3, 1:Ny-1]
    ) / (3 + Bi_g)

    # Flatten initial condition (column-major, MATLAB-compatible)
    y0 = T0.T.reshape(Nx * Ny)

    # System of ODEs
    def system(t, y):
        T = y.reshape((Ny, Nx)).T
        Tp = np.zeros_like(T)

        for i in range(1, Nx-1):
            for j in range(1, Ny-1):
                Tp[i, j] = alpha_g * (
                    T[i+1, j] + T[i-1, j]
                    + T[i, j+1] + T[i, j-1]
                    - 4 * T[i, j]
                )

        return Tp.T.reshape(Nx * Ny)

    # Time integration (stiff solver, equivalent to ode15s)
    sol = solve_ivp(
        system,
        (tau[0], tau[-1]),
        y0,
        t_eval=tau,
        method="BDF"
    )

    # Temperature field at final time
    T = sol.y[:, -1].reshape((Ny, Nx)).T

    # Temperature at the center of the bar
    i1 = N // 2
    i2 = M // 2
    Tc = sol.y[(i1 * Ny + i2), :]

    # Contour plot at final time
    plt.figure()
    plt.contourf(X, Y, T.T, cmap="jet")
    plt.colorbar()
    plt.colormap = "jet"
    plt.xlabel("x (cm)")
    plt.ylabel("y (cm)")
    plt.title(f"Temperature (°C) at t = {tau[-1]} s")

    # Temperature at the center vs time
    plt.figure()
    plt.plot(sol.t, Tc, "-bo")
    plt.xlabel("time (s)")
    plt.ylabel("Temperature (°C)")
    plt.title("Temperature at the center of the bar")

    # Combined plot
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.contourf(X, Y, T.T, cmap="jet")
    plt.colorbar()
    plt.xlabel("x (cm)")
    plt.ylabel("y (cm)")
    plt.title(f"Temperature (°C) at t = {tau[-1]} s")

    plt.subplot(1, 2, 2)
    plt.plot(sol.t, Tc, "-bo")
    plt.xlabel("time (s)")
    plt.ylabel("Temperature (°C)")
    plt.title("Temperature at the center of the bar")

    plt.tight_layout()
    filename = 'Figure_5_12.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

    return T, Tc